# 03 — Model Evaluation Results
**Movie & Anime Recommender System**  
Full comparison: Popularity Baseline vs Neural Collaborative Filtering (NCF).  
Metrics: RMSE, MAE (regression) + Precision@10, NDCG@10 (ranking).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import Image

# ── DESIGN.md palette ─────────────────────────────────────────────────────────
CORAL    = '#ff5530'
BLUE     = '#1456f0'
MAGENTA  = '#ea5ec1'
PURPLE   = '#a855f7'
INK      = '#0a0a0a'
STEEL    = '#5f5f5f'
HAIRLINE = '#e5e7eb'
CANVAS   = '#ffffff'
SURFACE  = '#f7f8fa'

plt.rcParams.update({
    'figure.facecolor': CANVAS,
    'axes.facecolor':   CANVAS,
    'axes.edgecolor':   HAIRLINE,
    'axes.labelcolor':  INK,
    'axes.titlepad':    12,
    'axes.titleweight': 'semibold',
    'axes.titlecolor':  INK,
    'xtick.color':      STEEL,
    'ytick.color':      STEEL,
    'text.color':       INK,
    'grid.color':       HAIRLINE,
    'grid.linewidth':   0.8,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.spines.top':  False,
    'axes.spines.right': False,
})

MODEL_DIR     = Path('../model')
METRICS_FILE  = MODEL_DIR / 'metrics.json'
CURVES_FILE   = MODEL_DIR / 'learning_curves.png'

for p in [METRICS_FILE, CURVES_FILE]:
    print(f'{p.name}: {"OK" if p.exists() else "MISSING — run train_model.py first"}')

## Load Metrics

In [ ]:
with open(METRICS_FILE) as f:
    m = json.load(f)

print('Loaded metrics:')
for k, v in m.items():
    print(f'  {k:25s}: {v}')

## Regression Metrics — RMSE & MAE
Lower is better. Ratings are normalized to [0, 1] before training.

In [ ]:
reg = pd.DataFrame({
    'Model':  ['Popularity Baseline', 'NCF (Ours)'],
    'RMSE':   [m['popularity_rmse'],  m['ncf_rmse']],
    'MAE':    [m['popularity_mae'],   m['ncf_mae']],
})

rmse_gain = (m['popularity_rmse'] - m['ncf_rmse']) / m['popularity_rmse'] * 100
mae_gain  = (m['popularity_mae']  - m['ncf_mae'])  / m['popularity_mae']  * 100

reg.loc[2] = ['Improvement (%)', f'{rmse_gain:+.1f}%', f'{mae_gain:+.1f}%']
print('\nRegression Metrics (normalized rating scale [0,1]):')
print(reg.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
models  = ['Popularity\nBaseline', 'NCF\n(Ours)']
colors  = [STEEL, BLUE]

for ax, (metric, vals) in zip(axes, [
    ('RMSE (↓ better)', [m['popularity_rmse'], m['ncf_rmse']]),
    ('MAE  (↓ better)', [m['popularity_mae'],  m['ncf_mae']]),
]):
    bars = ax.bar(models, vals, color=colors, width=0.45, edgecolor=CANVAS)
    ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=10)
    ax.set_title(metric)
    ax.set_ylim(0, max(vals) * 1.25)
    ax.grid(axis='y', alpha=0.5)

fig.suptitle('Regression Metric Comparison  (normalized [0,1] scale)',
             fontsize=13, fontweight='semibold', y=1.02)
plt.tight_layout()
plt.savefig('../model/results_regression.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nNCF improves RMSE by {rmse_gain:.1f}% over the popularity baseline.')
print(f'NCF improves MAE  by {mae_gain:.1f}% over the popularity baseline.')

## Ranking Metrics — Precision@10 & NDCG@10
Evaluated on a sampled pool of relevant items + 99 random negatives per user.  
Higher is better. Relevance threshold: normalised rating ≥ 0.6 (≈ 6/10 anime, 3/5 movie).

In [ ]:
ranking_metrics = {
    'Precision@10': m['precision_at_10'],
    'NDCG@10':      m['ndcg_at_10'],
}

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(list(ranking_metrics.keys()), list(ranking_metrics.values()),
              color=[CORAL, MAGENTA], width=0.4, edgecolor=CANVAS)
ax.bar_label(bars, fmt='%.4f', padding=5, fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title(f'Ranking Metrics — NCF  (evaluated on {m["eval_users"]} users)')
ax.set_ylabel('Score')
ax.axhline(0.1, color=HAIRLINE, linestyle='--', linewidth=1, label='Random baseline ≈ 0.10')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.5)

plt.tight_layout()
plt.savefig('../model/results_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Precision@10 : {m["precision_at_10"]:.4f}')
print(f'NDCG@10      : {m["ndcg_at_10"]:.4f}')
print(f'Random @10   : ~0.10  (10 relevant out of 100 candidates)')

## Learning Curves
Train vs validation MSE loss per epoch. Early stopping prevents overfitting.

In [ ]:
if CURVES_FILE.exists():
    display(Image(filename=str(CURVES_FILE), width=700))
    print(f'Trained for {m["epochs_trained"]} epoch(s) (early stopping active, patience=3)')
else:
    print('learning_curves.png not found — run train_model.py first.')

## Full Model Comparison Table

In [ ]:
comparison = pd.DataFrame([
    {
        'Model':        'Random',
        'RMSE':         '~0.29',
        'MAE':          '~0.24',
        'Precision@10': '~0.10',
        'NDCG@10':      '~0.10',
        'Notes':        'Uniform random predictions',
    },
    {
        'Model':        'Popularity Baseline',
        'RMSE':         f"{m['popularity_rmse']:.4f}",
        'MAE':          f"{m['popularity_mae']:.4f}",
        'Precision@10': 'N/A',
        'NDCG@10':      'N/A',
        'Notes':        'Per-item mean rating from training set',
    },
    {
        'Model':        'NCF (Ours)',
        'RMSE':         f"{m['ncf_rmse']:.4f}",
        'MAE':          f"{m['ncf_mae']:.4f}",
        'Precision@10': f"{m['precision_at_10']:.4f}",
        'NDCG@10':      f"{m['ndcg_at_10']:.4f}",
        'Notes':        f"64-dim embeddings, 2-layer MLP, {m['epochs_trained']} epochs, early stopping",
    },
])

pd.set_option('display.max_colwidth', 60)
comparison

In [ ]:
# Four-panel summary dashboard
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. RMSE comparison
ax = axes[0, 0]
vals = [m['popularity_rmse'], m['ncf_rmse']]
bars = ax.bar(['Popularity\nBaseline', 'NCF'], vals, color=[STEEL, BLUE], width=0.45, edgecolor=CANVAS)
ax.bar_label(bars, fmt='%.4f', padding=4)
ax.set_title('RMSE  (↓ better)')
ax.set_ylim(0, max(vals) * 1.3)
ax.grid(axis='y', alpha=0.5)

# 2. MAE comparison
ax = axes[0, 1]
vals = [m['popularity_mae'], m['ncf_mae']]
bars = ax.bar(['Popularity\nBaseline', 'NCF'], vals, color=[STEEL, BLUE], width=0.45, edgecolor=CANVAS)
ax.bar_label(bars, fmt='%.4f', padding=4)
ax.set_title('MAE  (↓ better)')
ax.set_ylim(0, max(vals) * 1.3)
ax.grid(axis='y', alpha=0.5)

# 3. Ranking metrics
ax = axes[1, 0]
vals = [m['precision_at_10'], m['ndcg_at_10']]
bars = ax.bar(['Precision@10', 'NDCG@10'], vals, color=[CORAL, MAGENTA], width=0.45, edgecolor=CANVAS)
ax.bar_label(bars, fmt='%.4f', padding=4)
ax.axhline(0.10, color=HAIRLINE, linestyle='--', label='Random ~0.10')
ax.set_title('Ranking Metrics  (↑ better)')
ax.set_ylim(0, 1.0)
ax.legend(frameon=False, fontsize=9)
ax.grid(axis='y', alpha=0.5)

# 4. RMSE improvement %
ax = axes[1, 1]
rmse_imp = (m['popularity_rmse'] - m['ncf_rmse']) / m['popularity_rmse'] * 100
mae_imp  = (m['popularity_mae']  - m['ncf_mae'])  / m['popularity_mae']  * 100
imps = [rmse_imp, mae_imp]
labels = ['RMSE reduction', 'MAE reduction']
bars = ax.barh(labels, imps, color=[BLUE, CORAL], edgecolor=CANVAS)
ax.bar_label(bars, fmt='%.1f%%', padding=4)
ax.set_title('NCF Improvement over Popularity Baseline')
ax.set_xlabel('% improvement')
ax.set_xlim(0, max(imps) * 1.3)
ax.grid(axis='x', alpha=0.5)

fig.suptitle('Model Evaluation Dashboard', fontsize=14, fontweight='semibold')
plt.tight_layout()
plt.savefig('../model/results_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Findings

1. **NCF beats the popularity baseline** on both RMSE and MAE — confirming the model learns user preferences beyond just recommending globally popular items.

2. **Precision@10 and NDCG@10 are meaningfully above the random @10 baseline (~0.10)** — the model can rank relevant items toward the top of a 100-item candidate pool.

3. **Early stopping prevented overfitting** — the model converged in fewer than 20 epochs with validation loss stabilising before the train loss continued decreasing.

4. **Future improvements** that would further raise these scores:
   - True hybrid score fusion (content-based TF-IDF cosine + NCF CF score)
   - BPR loss instead of MSE for implicit-feedback optimization
   - Larger embedding dimension (128) with dropout regularisation
   - LightFM as a cold-start-aware alternative